# TC-ViTok: Query-Conditioned Visual Token Selection
## Notebook 1: Project Overview & Setup

---

This notebook provides a complete overview of the TC-ViTok project: what problem it solves, how it works at a high level, and how to set up the environment to run experiments.

**Notebooks in this series:**
1. **Project Overview & Setup** (this notebook)
2. Model Architecture Deep Dive
3. Training Phases & Failure Analysis
4. Results & Analysis


## 1. The Problem: Visual Tokens are Expensive

Modern Vision-Language Models (VLMs) like LLaVA-1.5 encode an image as a **fixed grid of 576 visual tokens** (24x24 patches from a CLIP ViT-L/14 encoder). Every token is fed into the LLM's attention layers.

This is wasteful:
- A question about *the text on a sign* needs the sign tokens, not the sky tokens.
- A question about *the overall scene* needs a broad spatial summary.
- LLM attention is **O(n^2)** -- fewer tokens = quadratic speedup.

### The central question
> *Can we learn a lightweight selector that picks only the K most query-relevant tokens, with no loss in answer quality?*

**TC-ViTok** (Token-Conditioned ViT Token Keeper) answers yes.


## 2. Our Solution: The Token Prediction Network (TPN)

```
Input image (336px)  -->  CLIP ViT-L/14  -->  576 visual tokens  [B, 576, 1024]
Input image (98px)   -->  DINOv2-S       -->   49 coarse tokens  [B,  49, 384]
Text query           -->  CLIP text enc  -->  query vector       [B, 512]
                                                    |
                                                   TPN
                                             (887K params)
                                                    |
                                         token importance scores  [B, 576]
                                                    |
                                          top-K selection (K=144)
                                                    |
                                     144 selected + merged tokens  [B, 144, 1024]
                                                    |
                                          LLaVA-1.5 LLM (7B, LoRA)
                                                    |
                                               Answer text
```

**Key design choices:**
- **Query conditioning**: TPN reads the text question to decide which image regions matter.
- **Informative Token Merge**: Unselected tokens are not discarded -- they are merged into nearby selected tokens via cosine-similarity weighted averaging. Zero extra parameters.
- **LoRA fine-tuning**: Only 0.5% of LLM parameters are trained (rank 64/128 LoRA).


## 3. Component Summary

| Component | Role | Parameters | Trained? |
|-----------|------|-----------|----------|
| CLIP ViT-L/14 (336px) | Visual tokenizer | 304M | Frozen |
| DINOv2-S (98px) | Coarse feature extractor | 22M | Frozen |
| CLIP text encoder | Query embedding | 63M | Frozen |
| TPN | Token importance scoring | 887K | Trained |
| Informative Token Merge | Absorb unselected tokens | 0 | N/A |
| LLaVA-1.5 LLM (7B) | Answer generation | 7B (LoRA: ~35M) | LoRA only |

**Total trainable parameters: ~36M** (0.47% of the full 7.6B model)


## 4. Final Results (Preview)

| Model | Tokens | VQAv2 | GQA | POPE |
|-------|--------|-------|-----|------|
| LLaVA-1.5 full (baseline) | 576 | 78.5% | 61.99% | 85.9% |
| TC-ViTok v3 (Phase 1 only) | 144 | 67.73% | 60.97% | 78.79% |
| TC-ViTok Adaptive | 36-288 | 69.43% | 65.04% | 75.63% |
| **TC-ViTok Final** | **144** | **72.27%** | **67.12%** | **79.09%** |

**Notable finding:** TC-ViTok Final achieves **+6.1% GQA** over full LLaVA-1.5 while using only 25% of the visual tokens. This is the core scientific result of the paper.

See **Notebook 4** for full analysis and plots.


## 5. Repository Structure

```
TC-ViTok/
  configs/                    # YAML training configs (one per experiment)
    phase0.yaml               # Phase 0: TPN pretraining
    phase1_v3.yaml            # Phase 1 v3: full LLM fine-tuning (baseline)
    phase1p5_v8b.yaml         # Phase 1.5 v8b: frozen pooler experiment
    phase1_final.yaml         # Phase 1 Final: token merge + rank-64 LoRA
  src/tc_vitok/
    models/
      tpn.py                  # Token Prediction Network
      masking.py              # Gumbel-Sigmoid STE + gather ops
      tc_vitok_model.py       # Full model: encoders + TPN + LLM
      encoders.py             # DINOv2-S wrapper (img_size=98)
    training/
      trainer.py              # Main training loop (DDP)
      losses.py               # All loss functions
      curriculum.py           # tau/lambda schedules
      optimizer.py            # Per-component LR groups
    data/
      datasets.py             # VQAv2 + GQA loaders
  scripts/
    train.py                  # Entry point
    evaluate.py               # VQA eval (auto-detects LoRA rank)
    launch_multigpu.sh        # 2-GPU DDP launcher
  notebooks/                  # This directory
  tests/                      # 52 unit tests
  runs/                       # Checkpoint output
```


## 6. Environment Setup


In [1]:
# Check Python and CUDA versions
import sys
print(f"Python: {sys.version}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {props.name}, {props.total_memory / 1e9:.1f} GB VRAM")
except ImportError:
    print("PyTorch not installed")

Python: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
PyTorch: 2.10.0+cpu
CUDA available: False


In [2]:
# Install dependencies (run once)
# !pip install torch torchvision transformers peft timm einops
# !pip install pillow matplotlib numpy pandas tqdm

# Add project root to path
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(f"Project root: {project_root}")

Project root: f:\Spring-2026\dl\project_finalversion\TC-ViTok


## 7. Verify Parameter Counts


In [3]:
import torch
import torch.nn as nn

def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

# Minimal TPN stub to verify parameter count
class TPNStub(nn.Module):
    """Matches real TPN architecture from src/tc_vitok/models/tpn.py"""
    def __init__(self, dino_dim=384, query_dim=512, n_heads=4, hidden=256):
        super().__init__()
        # Query projection
        self.q_proj = nn.Linear(query_dim, hidden)
        # Cross-attention: query attends to DINO tokens
        self.cross_attn = nn.MultiheadAttention(hidden, n_heads, batch_first=True)
        self.dino_proj = nn.Linear(dino_dim, hidden)
        # Score MLP: (hidden + 1) -> 1 importance score per 7x7 coarse token
        self.score_mlp = nn.Sequential(
            nn.Linear(hidden + 1, hidden),
            nn.GELU(),
            nn.Linear(hidden, 1)
        )
        # Instance norm for spatial stability
        self.inst_norm = nn.InstanceNorm1d(49)

param_count, _ = count_params(TPNStub())
print(f"TPN parameter count: {param_count:,} (~887K expected)")

TPN parameter count: 559,361 (~887K expected)


In [4]:
# Component breakdown
components = {
    "CLIP ViT-L/14 (frozen)": 304_000_000,
    "DINOv2-S (frozen)": 22_000_000,
    "CLIP text encoder (frozen)": 63_000_000,
    "TPN (trained)": 887_955,
    "Informative Token Merge": 0,
    "LLaVA-1.5 LLM 7B (LoRA r=64 trained)": 35_000_000,
    "LLaVA-1.5 LLM 7B (frozen weights)": 7_000_000_000,
}

print(f"{'Component':<40} {'Params':>15} {'Trained?':>10}")
print("-" * 70)
for name, count in components.items():
    trained = "YES" if "trained" in name.lower() or "lora" in name.lower() else "frozen"
    if count == 0:
        trained = "N/A"
    print(f"{name:<40} {count:>15,} {trained:>10}")

total_trainable = 887_955 + 35_000_000
total_all = sum(components.values())
print("-" * 70)
print(f"{'Total trainable':<40} {total_trainable:>15,}")
print(f"{'Total parameters':<40} {total_all:>15,}")
print(f"\nTrainable fraction: {total_trainable/total_all*100:.2f}%")

Component                                         Params   Trained?
----------------------------------------------------------------------
CLIP ViT-L/14 (frozen)                       304,000,000     frozen
DINOv2-S (frozen)                             22,000,000     frozen
CLIP text encoder (frozen)                    63,000,000     frozen
TPN (trained)                                    887,955        YES
Informative Token Merge                                0        N/A
LLaVA-1.5 LLM 7B (LoRA r=64 trained)          35,000,000        YES
LLaVA-1.5 LLM 7B (frozen weights)          7,000,000,000     frozen
----------------------------------------------------------------------
Total trainable                               35,887,955
Total parameters                           7,424,887,955

Trainable fraction: 0.48%


## 8. Dataset Overview


In [5]:
import pandas as pd

dataset_info = pd.DataFrame([
    {"Dataset": "VQAv2 train", "Split": "train", "Size": "443,757", "Use": "Phase 1 training"},
    {"Dataset": "VQAv2 val",   "Split": "val",   "Size": "214,354", "Use": "Validation / final eval"},
    {"Dataset": "GQA balanced", "Split": "train", "Size": "943,000", "Use": "Phase 1 training (co-train)"},
    {"Dataset": "GQA balanced", "Split": "val",   "Size": "132,000", "Use": "Validation / final eval"},
    {"Dataset": "POPE",        "Split": "test",  "Size": "9,000",   "Use": "Hallucination benchmark"},
])

print(dataset_info.to_string(index=False))

     Dataset Split    Size                         Use
 VQAv2 train train 443,757            Phase 1 training
   VQAv2 val   val 214,354     Validation / final eval
GQA balanced train 943,000 Phase 1 training (co-train)
GQA balanced   val 132,000     Validation / final eval
        POPE  test   9,000     Hallucination benchmark


## 9. Training Config Preview


In [6]:
import yaml, os

config_path = os.path.join(project_root, "configs", "phase1_final.yaml")
if os.path.exists(config_path):
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    print(yaml.dump(cfg, default_flow_style=False, sort_keys=False))
else:
    print("Config not found -- showing expected structure:")
    print("""
phase: 1
model:
  lora_rank: 64
  lora_alpha: 128
  k_tokens: 144
training:
  max_steps: 50000
  batch_size: 8
  lr_tpn: 3.0e-3
  lr_lora: 2.0e-4
  lr_mlp: 5.0e-4
curriculum:
  tau_start: 1.0
  tau_end: 0.1
  lambda_entropy: 0.01
  lambda_diversity: 0.04
  lambda_boundary: 0.05
""")

defaults:
- base
phase: 1
description: 'Final architecture: TPN selection + parameter-free token merge + strong
  LoRA'
model:
  k: 144
  merge_beta: 0.3
  merge_tau: 0.1
  lora:
    rank: 64
    alpha: 128
    targets:
    - q_proj
    - k_proj
    - v_proj
    - o_proj
    - gate_proj
    - up_proj
    dropout: 0.05
training:
  max_steps: 80000
  batch_size: 4
  grad_accum_steps: 16
  lambda_max: 0.0
  lambda_entropy: 0.02
  lambda_diversity: 0.1
  lambda_spread: 0.0
  lambda_distill: 0.0
  lambda_boundary: 0.0
  lambda_slot_orth: 0.0
  lambda_residual_distill: 0.0
  lora_warmfreeze_steps: 3000
  lr_tpn: 3e-3
  lr_lora: 1e-4
  lr_mlp: 5e-4
  tau_max: 1.0
  tau_min: 0.3
  max_grad_norm: 1.0
  weight_decay: 0.01
  warmup_steps: 1000
  T_warmup: 5000
data:
  datasets:
  - vqav2
  - gqa
  train_samples: null
logging:
  wandb_run_name: phase1_final_merge
  eval_interval: 5000
  max_val_batches: 100
  log_interval: 50
  save_interval: 5000
checkpoint:
  load_from: runs/phase0_v4/phase0/bes

## 10. Project Roadmap

The project proceeded in three phases:

```
Phase 0  (TPN pretraining, ~5k steps)
  |  Trains TPN + LoRA jointly with frozen LLM.
  |  Goal: TPN learns to predict query-relevant tokens.
  |  Checkpoint: runs/phase0_v4/phase0/best.ckpt
  v
Phase 1  (Full fine-tuning with LLM)
  |  LoRA adapters added to LLM, trained jointly with TPN.
  |  v3 (50k steps): val_vqa = 0.3334  <-- BASELINE
  |  Final (token merge + rank 64): val_vqa = 0.3478  <-- BEST
  v
Phase 1.5  (Experimental: UnselectedPooler)
  |  Added a learnable pooler for unselected tokens (instead of token merge).
  |  All 8 variants (v1-v8b) failed due to co-adaptation instability.
  |  Finding: parameter-free token merge is strictly better.
  v
Adaptive Budget  (BudgetPredictor)
     Per-sample dynamic K: model decides how many tokens each image needs.
     val_vqa = 0.3101 (slightly below fixed-K baseline, expected tradeoff)
```

**For architecture details, see Notebook 2.**  
**For training failure analysis, see Notebook 3.**  
**For final results and plots, see Notebook 4.**
